# Notebook 07: Panel Regressions & Panel VAR
This notebook implements the core econometric regressions: disaggregated policy pressure regressions (Labor vs. Community), ESEA waiver interaction models ($H_7$), and a Helmert-transformed GMM Panel VAR to Granger-test feedback loops while correcting for Nickell bias.

### Important Project Safety Notice

Before executing or citing the findings in this notebook, please read the public guidance on what this project is and is not claiming:  

[docs/not_saying.md](../docs/not_saying.md) - *What This Theory Is NOT Claiming*

## 1. Library Imports & Data Realignment
Load state-year panel data and merge disaggregated weights from the ESSA coding sheet.

In [1]:
import pandas as pd
import numpy as np

# Load state-year panel data (which contains real political and waiver variables)
df = pd.read_csv('../data/processed/state_year_panel.csv')
df_essa = pd.read_csv('../data/raw/essa_plan_coding_sheet.csv')

# Merge disaggregated weights only if they don't already exist in df
if 'academic_achievement_weight' not in df.columns:
    df = df.merge(df_essa[['state', 'academic_achievement_weight', 'academic_growth_weight']], on='state', how='left')
else:
    print('academic_achievement_weight already exists in df, skipping merge.')

# Verify we have has_waiver and political controls loaded directly from panel
print('Data aligned. Sample size:', df.shape)
print('Waiver states:', list(df[df['has_waiver'] == 1]['state'].unique()[:5]))
print('Trifecta years:', df['trifecta'].sum())

Data aligned. Sample size: (765, 54)
Waiver states: ['AK', 'AL', 'AR', 'AZ', 'CO']
Trifecta years: 377


## 2. Construct Disaggregated Policy Pressure Indices
We construct separate indices for:
*   **Community-directed pressure** (parent-salient): uses pre-ESSA policy intensity, and post-ESSA academic achievement weight.
*   **Labor-directed pressure** (teacher-salient): uses pre-ESSA policy intensity, and post-ESSA academic growth weight.

In [2]:
df['academic_achievement_weight'] = df['academic_achievement_weight'].fillna(40.0)
df['academic_growth_weight'] = df['academic_growth_weight'].fillna(40.0)

df['raw_labor'] = df['vam_eval']
df['raw_comm'] = df['exit_exam'] + df['af_grading'] + df['third_grade_retention']

# Scale post-ESSA weights before standardizing within-era
mask_post = df['year'] >= 2018
df.loc[mask_post, 'raw_comm'] = df.loc[mask_post, 'raw_comm'] * (df.loc[mask_post, 'academic_achievement_weight'] / 100.0)
df.loc[mask_post, 'raw_labor'] = df.loc[mask_post, 'raw_labor'] * (df.loc[mask_post, 'academic_growth_weight'] / 100.0)

# Standardize within-era
df['policy_community'] = 0.0
df['policy_labor'] = 0.0

for mask in [df['year'] <= 2017, df['year'] >= 2018]:
    for col, target in [('raw_comm', 'policy_community'), ('raw_labor', 'policy_labor')]:
        m = df.loc[mask, col].mean()
        s = df.loc[mask, col].std()
        if pd.isna(s) or s == 0: s = 1.0
        df.loc[mask, target] = (df.loc[mask, col] - m) / s

print('Disaggregated policy indices constructed successfully.')


Disaggregated policy indices constructed successfully.


C:\Users\admir\AppData\Local\Temp\ipykernel_9516\3791847699.py:9: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[0.    0.    0.    0.    0.    0.    0.    0.4   0.4   0.4   0.4   0.4
 0.4   0.4   0.    0.    0.    0.    0.    0.    0.    0.6   0.6   0.6
 0.6   0.6   0.6   0.6   0.    0.    0.    0.    0.    0.    0.    0.3
 0.3   0.3   0.3   0.3   0.3   0.3   0.    0.    0.    0.    0.    0.
 0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
 0.    0.    0.    0.999 0.999 0.999 0.999 0.999 0.999 0.999 0.3   0.3
 0.3   0.3   0.3   0.3   0.3   0.    0.    0.    0.    0.    0.    0.
 0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
 0.    0.    0.    0.    0.    0.    0.    0.    0.    0.8   0.8   0.8
 0.8   0.8   0.8   0.8   0.    0.    0.    0.    0.    0.    0.    0.
 0.    0.    0.    0.    0.    0.    0.4   0.4   0.4   0.4   0.4   0.4
 0.4   0.3   0.3   0.3   0.3 

## 3. Disaggregated Backlash Regressions
We estimate the impact of our disaggregated policy indices on backlash scores using OLS with state and year fixed effects and cluster-robust standard errors.

In [3]:
# Create lagged variables within each state
df['policy_lag1'] = df.groupby('state')['policy_intensity'].shift(1)
df['policy_comm_lag1'] = df.groupby('state')['policy_community'].shift(1)
df['policy_labor_lag1'] = df.groupby('state')['policy_labor'].shift(1)

from linearmodels.panel import PanelOLS

# Drop NaNs to prevent panel regression index alignment issues
cols_p = ['state', 'year', 'backlash', 'policy_lag1', 'policy_comm_lag1', 'policy_labor_lag1', 'gov_party_rep', 'trifecta', 'election_year']
df_p = df[cols_p].dropna().set_index(['state', 'year'])

# Model 1: Baseline Policy on Backlash
fit1 = PanelOLS.from_formula(
    'backlash ~ policy_lag1 + gov_party_rep + trifecta + election_year + EntityEffects + TimeEffects',
    data=df_p
).fit(cov_type='clustered', cluster_entity=True)
print('=== Baseline Policy on Backlash (PanelOLS) ===')
print(fit1.summary.tables[1])

# Model 2: Community-Directed Policy on Backlash
fit2 = PanelOLS.from_formula(
    'backlash ~ policy_comm_lag1 + gov_party_rep + trifecta + election_year + EntityEffects + TimeEffects',
    data=df_p
).fit(cov_type='clustered', cluster_entity=True)
print('\n=== Community-Directed Policy on Backlash (PanelOLS) ===')
print(fit2.summary.tables[1])

# Model 3: Labor-Directed Policy on Backlash
fit3 = PanelOLS.from_formula(
    'backlash ~ policy_labor_lag1 + gov_party_rep + trifecta + election_year + EntityEffects + TimeEffects',
    data=df_p
).fit(cov_type='clustered', cluster_entity=True)
print('\n=== Labor-Directed Policy on Backlash (PanelOLS) ===')
print(fit3.summary.tables[1])


=== Baseline Policy on Backlash (PanelOLS) ===
                               Parameter Estimates                               
               Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
---------------------------------------------------------------------------------
policy_lag1      -0.1049     0.1148    -0.9136     0.3613     -0.3303      0.1206
gov_party_rep    -0.2130     0.1197    -1.7798     0.0756     -0.4480      0.0220
trifecta         -1.0115     0.5151    -1.9636     0.0500     -2.0230   3.223e-05
election_year     0.0478     0.0369     1.2940     0.1961     -0.0247      0.1203

=== Community-Directed Policy on Backlash (PanelOLS) ===
                                Parameter Estimates                                 
                  Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------------
policy_comm_lag1    -0.1083     0.0925    -1.1709     0.2421     -

## 4. ESEA Waiver Dampening Interactions ($H_7$)
We test if ESEA waiver status (institutional lock-in) buffers the policy-backlash link (H7a) and the backlash-correction link (H7b).

In [4]:
# Lagged variables for H7 interaction checks
df['backlash_lag1'] = df.groupby('state')['backlash'].shift(1)
df['delta_policy'] = df.groupby('state')['policy_intensity'].diff()

# Explicitly construct interaction terms to prevent index/parsing issues
df['policy_x_waiver'] = df['policy_lag1'] * df['has_waiver']
df['backlash_x_waiver'] = df['backlash_lag1'] * df['has_waiver']

df['gov_party_change'] = df.groupby('state')['gov_party_rep'].diff().abs().fillna(0.0)
df['backlash_x_rep'] = df['backlash_lag1'] * df['gov_party_rep']

cols_h7 = ['state', 'year', 'backlash', 'policy_lag1', 'has_waiver', 'policy_x_waiver', 'gov_party_rep', 'trifecta', 'delta_policy', 'backlash_lag1', 'backlash_x_waiver', 'gov_party_change', 'backlash_x_rep']
df_h7 = df[cols_h7].dropna().set_index(['state', 'year'])

# H7a: Backlash Dampening
fit_h7a = PanelOLS.from_formula(
    'backlash ~ policy_lag1 + has_waiver + policy_x_waiver + gov_party_rep + trifecta + EntityEffects + TimeEffects',
    data=df_h7
).fit(cov_type='clustered', cluster_entity=True)
print('=== H7a: Backlash Dampening (ESEA Waiver interaction) ===')
print(fit_h7a.summary.tables[1])

# H7b: Correction Dampening
fit_h7b = PanelOLS.from_formula(
    'delta_policy ~ backlash_lag1 + has_waiver + backlash_x_waiver + gov_party_rep + trifecta + EntityEffects + TimeEffects',
    data=df_h7
).fit(cov_type='clustered', cluster_entity=True)
print('\n=== H7b: Correction Dampening (ESEA Waiver interaction) ===')
print(fit_h7b.summary.tables[1])

# H7b: Partisan interaction model (competitor hypothesis)
fit_partisan = PanelOLS.from_formula(
    'delta_policy ~ backlash_lag1 + gov_party_change + backlash_x_rep + gov_party_rep + trifecta + EntityEffects + TimeEffects',
    data=df_h7
).fit(cov_type='clustered', cluster_entity=True)
print('\n=== Competitor Hypothesis: Partisan Cycling and Backlash Rollback ===')
print(fit_partisan.summary.tables[1])


=== H7a: Backlash Dampening (ESEA Waiver interaction) ===
                                Parameter Estimates                                
                 Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
-----------------------------------------------------------------------------------
policy_lag1        -0.1292     0.1255    -1.0297     0.3035     -0.3756      0.1172
has_waiver          0.0996     0.1783     0.5589     0.5764     -0.2505      0.4497
policy_x_waiver     0.0471     0.0810     0.5813     0.5612     -0.1120      0.2063
gov_party_rep      -0.2233     0.1183    -1.8874     0.0596     -0.4556      0.0090
trifecta           -1.0141     0.5038    -2.0132     0.0445     -2.0033     -0.0249



=== H7b: Correction Dampening (ESEA Waiver interaction) ===
                                 Parameter Estimates                                 
                   Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
-------------------------------------------------------------------------------------
backlash_lag1         0.1092     0.0322     3.3916     0.0007      0.0460      0.1724
has_waiver            0.6365     0.0615     10.356     0.0000      0.5158      0.7571
backlash_x_waiver    -0.1685     0.0433    -3.8940     0.0001     -0.2535     -0.0835
gov_party_rep         0.0405     0.0330     1.2287     0.2196     -0.0242      0.1052
trifecta              0.0682     0.0615     1.1087     0.2680     -0.0526      0.1891

=== Competitor Hypothesis: Partisan Cycling and Backlash Rollback ===
                                Parameter Estimates                                 
                  Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
----------

## 5. Helmert-Transformed Panel VAR
To eliminate Nickell bias ($1/T$) in our panel, we apply the forward orthogonal deviations (Helmert transformation) to our variables, then estimate the vector autoregression.

In [5]:
def helmert_transform(df, cols):
    df_transformed = []
    for state, group in df.groupby('state'):
        group = group.sort_values('year').copy()
        T = len(group)
        for t in range(T - 1):
            row = group.iloc[t].copy()
            for col in cols:
                forward_vals = group.iloc[t+1:][col].values
                mean_forward = np.mean(forward_vals)
                scale = np.sqrt((T - t - 1) / (T - t))
                row[col] = scale * (group.iloc[t][col] - mean_forward)
            df_transformed.append(row)
    return pd.DataFrame(df_transformed)

# Estimating a 2-variable Panel VAR using IV/2SLS (Arellano-Bover GMM-style) to prevent boundary leaks & Nickell bias
# 1. Prepare variables and create lagged levels on untransformed data
df_var_input = df[['state', 'year', 'policy_intensity', 'backlash']].dropna().copy()
df_var_input['L1_level_policy'] = df_var_input.groupby('state')['policy_intensity'].shift(1)
df_var_input['L1_level_backlash'] = df_var_input.groupby('state')['backlash'].shift(1)

# 2. Transform the dependent and regressor variables
df_helmert = helmert_transform(df_var_input, ['policy_intensity', 'backlash'])

# 3. Shift the transformed variables to get the regressors
df_helmert['L1_policy_intensity'] = df_helmert.groupby('state')['policy_intensity'].shift(1)
df_helmert['L1_backlash'] = df_helmert.groupby('state')['backlash'].shift(1)

df_var_clean = df_helmert.dropna(subset=['L1_policy_intensity', 'L1_backlash', 'L1_level_policy', 'L1_level_backlash']).copy()

# 4. Fit equation-by-equation IV2SLS, using L2 levels as instruments for L1 transformed regressors
from linearmodels.iv import IV2SLS

print('=== Equation 1: policy_intensity ~ L1_policy_intensity + L1_backlash (Instrumented by L1 levels) ===')
res_eq1 = IV2SLS.from_formula(
    'policy_intensity ~ 1 + [L1_policy_intensity + L1_backlash ~ L1_level_policy + L1_level_backlash]',
    data=df_var_clean
).fit(cov_type='clustered', clusters=df_var_clean['state'])
print(res_eq1.summary.tables[1])

print('\n=== Equation 2: backlash ~ L1_policy_intensity + L1_backlash (Instrumented by L1 levels) ===')
res_eq2 = IV2SLS.from_formula(
    'backlash ~ 1 + [L1_policy_intensity + L1_backlash ~ L1_level_policy + L1_level_backlash]',
    data=df_var_clean
).fit(cov_type='clustered', clusters=df_var_clean['state'])
print(res_eq2.summary.tables[1])


=== Equation 1: policy_intensity ~ L1_policy_intensity + L1_backlash (Instrumented by L1 levels) ===
                                  Parameter Estimates                                  
                     Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
---------------------------------------------------------------------------------------
Intercept               0.0244     0.0161     1.5104     0.1309     -0.0073      0.0560
L1_policy_intensity     0.5906     0.0771     7.6645     0.0000      0.4396      0.7416
L1_backlash             0.0549     0.0257     2.1341     0.0328      0.0045      0.1054

=== Equation 2: backlash ~ L1_policy_intensity + L1_backlash (Instrumented by L1 levels) ===
                                  Parameter Estimates                                  
                     Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
---------------------------------------------------------------------------------------
Intercept    

## 6. Save the Regressions Outputs
We save the updated dataframe back to disk to verify the columns exist for visualization.

In [6]:
df.to_csv('../data/processed/state_year_panel.csv', index=False)
print('Panel dataset updated with regression columns.')

Panel dataset updated with regression columns.


## 7. Robustness Checks & Sensitivity Analysis

We execute a series of pre-registered robustness checks to test the stability of our baseline panel OLS regressions under alternative specifications, lag lengths, sub-indicators, and sample partitions.

In [7]:
from linearmodels.panel import PanelOLS
import statsmodels.formula.api as smf

# Prepare additional variables
df['policy_lag2'] = df.groupby('state')['policy_intensity'].shift(2)
df['raw_policy_lag1'] = df.groupby('state')['raw_intensity'].shift(1)
df['policy_standardized_whole'] = (df['raw_intensity'] - df['raw_intensity'].mean()) / df['raw_intensity'].std()
df['policy_whole_lag1'] = df.groupby('state')['policy_standardized_whole'].shift(1)

cols_rob = ['state', 'year', 'backlash', 'backlash_media', 'backlash_mass', 'policy_intensity', 'policy_lag1', 'policy_lag2', 'raw_policy_lag1', 'policy_whole_lag1', 'gov_party_rep', 'trifecta', 'election_year']
df_rob = df[cols_rob].dropna().copy()

print('=== Robustness Check 1: Alternative Lag Lengths (2-Year Lag) ===')
fit_lag2 = PanelOLS.from_formula(
    'backlash ~ policy_lag2 + gov_party_rep + trifecta + election_year + EntityEffects + TimeEffects',
    data=df_rob.set_index(['state', 'year'])
).fit(cov_type='clustered', cluster_entity=True)
print(fit_lag2.summary.tables[1])

print('\n=== Robustness Check 2: Alternative Backlash Measures (Media Only) ===')
fit_media = PanelOLS.from_formula(
    'backlash_media ~ policy_lag1 + gov_party_rep + trifecta + election_year + EntityEffects + TimeEffects',
    data=df_rob.set_index(['state', 'year'])
).fit(cov_type='clustered', cluster_entity=True)
print(fit_media.summary.tables[1])

print('\n=== Robustness Check 3: Alternative Backlash Measures (Mass Only) ===')
fit_mass = PanelOLS.from_formula(
    'backlash_mass ~ policy_lag1 + gov_party_rep + trifecta + election_year + EntityEffects + TimeEffects',
    data=df_rob.set_index(['state', 'year'])
).fit(cov_type='clustered', cluster_entity=True)
print(fit_mass.summary.tables[1])

print('\n=== Robustness Check 4: Alternative Policy Intensity (Raw Index) ===')
fit_raw = PanelOLS.from_formula(
    'backlash ~ raw_policy_lag1 + gov_party_rep + trifecta + election_year + EntityEffects + TimeEffects',
    data=df_rob.set_index(['state', 'year'])
).fit(cov_type='clustered', cluster_entity=True)
print(fit_raw.summary.tables[1])

print('\n=== Robustness Check 5: State-Specific Linear Trends ===')
fit_trends = smf.ols(
    'backlash ~ policy_lag1 + gov_party_rep + trifecta + election_year + C(state) + C(year) + C(state):year',
    data=df_rob
).fit(cov_type='cluster', cov_kwds={'groups': df_rob['state']})
print('Policy Lag 1 coefficient:', fit_trends.params['policy_lag1'], 'p-value:', fit_trends.pvalues['policy_lag1'])

print('\n=== Robustness Check 6: Excluding High-Leverage States (FL, TX, NY, WA) ===')
df_ex = df_rob[~df_rob['state'].isin(['FL', 'TX', 'NY', 'WA'])].set_index(['state', 'year'])
fit_ex = PanelOLS.from_formula(
    'backlash ~ policy_lag1 + gov_party_rep + trifecta + election_year + EntityEffects + TimeEffects',
    data=df_ex
).fit(cov_type='clustered', cluster_entity=True)
print(fit_ex.summary.tables[1])

print('\n=== Robustness Check 7: Pre-ESSA Split (<= 2017) ===')
df_pre = df_rob[df_rob['year'] <= 2017].set_index(['state', 'year'])
fit_pre = PanelOLS.from_formula(
    'backlash ~ policy_lag1 + gov_party_rep + trifecta + election_year + EntityEffects + TimeEffects',
    data=df_pre
).fit(cov_type='clustered', cluster_entity=True)
print(fit_pre.summary.tables[1])

print('\n=== Robustness Check 8: Post-ESSA Split (>= 2018) ===')
df_post = df_rob[df_rob['year'] >= 2018].set_index(['state', 'year'])
fit_post = PanelOLS.from_formula(
    'backlash ~ policy_lag1 + gov_party_rep + trifecta + election_year + EntityEffects + TimeEffects',
    data=df_post
).fit(cov_type='clustered', cluster_entity=True)
print(fit_post.summary.tables[1])

print('\n=== Robustness Check 9: Sensitivity to Whole Sample Standardization ===')
fit_whole = PanelOLS.from_formula(
    'backlash ~ policy_whole_lag1 + gov_party_rep + trifecta + election_year + EntityEffects + TimeEffects',
    data=df_rob.set_index(['state', 'year'])
).fit(cov_type='clustered', cluster_entity=True)
print(fit_whole.summary.tables[1])

print('\n=== Robustness Check 10: Clustered Bootstrap Confidence Intervals ===')
states_list = df['state'].unique()
bootstrap_coefs = []
np.random.seed(42)
for b in range(100):
    resampled_states = np.random.choice(states_list, size=len(states_list), replace=True)
    resampled_df_list = []
    for idx, s in enumerate(resampled_states):
        state_data = df[df['state'] == s].copy()
        state_data['bootstrap_id'] = f'{s}_{idx}'
        resampled_df_list.append(state_data)
    resampled_df = pd.concat(resampled_df_list, ignore_index=True)
    resampled_df['policy_lag1'] = resampled_df.groupby('bootstrap_id')['policy_intensity'].shift(1)
    df_boot_p = resampled_df[['bootstrap_id', 'year', 'backlash', 'policy_lag1', 'gov_party_rep', 'trifecta', 'election_year']].dropna()
    fit_boot = smf.ols(
        'backlash ~ policy_lag1 + gov_party_rep + trifecta + election_year + C(bootstrap_id) + C(year)',
        data=df_boot_p
    ).fit()
    bootstrap_coefs.append(fit_boot.params['policy_lag1'])
bootstrap_coefs = np.array(bootstrap_coefs)
ci_lower = np.percentile(bootstrap_coefs, 2.5)
ci_upper = np.percentile(bootstrap_coefs, 97.5)
print(f'Policy Lag 1 Clustered Bootstrap 95% CI: [{ci_lower:.4f}, {ci_upper:.4f}]')


=== Robustness Check 1: Alternative Lag Lengths (2-Year Lag) ===
                               Parameter Estimates                               
               Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
---------------------------------------------------------------------------------
policy_lag2      -0.1001     0.1113    -0.9002     0.3684     -0.3186      0.1184
gov_party_rep    -0.1903     0.1323    -1.4391     0.1506     -0.4501      0.0694
trifecta         -1.1411     0.6340    -1.7998     0.0724     -2.3863      0.1041
election_year     0.0611     0.0521     1.1747     0.2406     -0.0411      0.1634

=== Robustness Check 2: Alternative Backlash Measures (Media Only) ===


                               Parameter Estimates                               
               Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
---------------------------------------------------------------------------------
policy_lag1      -0.0865     0.0888    -0.9738     0.3306     -0.2609      0.0879
gov_party_rep    -0.0703     0.0870    -0.8083     0.4192     -0.2412      0.1005
trifecta         -0.1617     0.0866    -1.8658     0.0626     -0.3318      0.0085
election_year    -0.0109     0.0674    -0.1617     0.8716     -0.1432      0.1214

=== Robustness Check 3: Alternative Backlash Measures (Mass Only) ===
                               Parameter Estimates                               
               Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
---------------------------------------------------------------------------------
policy_lag1      -0.0488     0.0685    -0.7121     0.4767     -0.1834      0.0858
gov_party_rep    -0.0964   

Policy Lag 1 coefficient: -0.09942961615102516 p-value: 0.32815686096456387

=== Robustness Check 6: Excluding High-Leverage States (FL, TX, NY, WA) ===
                               Parameter Estimates                               
               Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
---------------------------------------------------------------------------------
policy_lag1      -0.1936     0.1136    -1.7048     0.0888     -0.4167      0.0295
gov_party_rep    -0.0970     0.1294    -0.7498     0.4537     -0.3511      0.1571
trifecta         -0.3253     0.1937    -1.6791     0.0937     -0.7057      0.0552
election_year     0.0598     0.0499     1.1980     0.2314     -0.0383      0.1579

=== Robustness Check 7: Pre-ESSA Split (<= 2017) ===


                               Parameter Estimates                               
               Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
---------------------------------------------------------------------------------
policy_lag1      -0.2018     0.1227    -1.6451     0.1012     -0.4434      0.0398
gov_party_rep    -0.2219     0.2413    -0.9193     0.3588     -0.6972      0.2535
trifecta         -0.5735     0.4390    -1.3064     0.1926     -1.4382      0.2912
election_year     0.0721     0.1061     0.6797     0.4974     -0.1369      0.2812

=== Robustness Check 8: Post-ESSA Split (>= 2018) ===
                               Parameter Estimates                               
               Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
---------------------------------------------------------------------------------
policy_lag1       0.0934     0.0972     0.9602     0.3378     -0.0980      0.2847
gov_party_rep    -0.0074     0.0718    -0.1

Policy Lag 1 Clustered Bootstrap 95% CI: [-0.3260, 0.1226]


## 8. Peer Panel Robustness and Theoretical Enhancements
Here, we implement the expert panel's recommendations: Leave-One-Component-Out (LOCO) tests, individual outcome components, disaggregated backlash variables, conditional placebos (1,000 runs), bootstrapped interaction models, policy-norm gap regressions, and rollback correction models with mean reversion controls.

In [8]:
# 1. LOCO Test (policy intensity without VAM) and re-run H7b
df['raw_intensity_no_vam'] = df['exit_exam'] + df['af_grading'] + df['third_grade_retention']
df['policy_intensity_no_vam'] = 0.0
for mask in [df['year'] <= 2017, df['year'] >= 2018]:
    mean_val = df.loc[mask, 'raw_intensity_no_vam'].mean()
    std_val = df.loc[mask, 'raw_intensity_no_vam'].std()
    if std_val == 0 or pd.isna(std_val): std_val = 1.0
    df.loc[mask, 'policy_intensity_no_vam'] = (df.loc[mask, 'raw_intensity_no_vam'] - mean_val) / std_val

df['delta_policy_no_vam'] = df.groupby('state')['policy_intensity_no_vam'].diff()
df['policy_no_vam_lag1'] = df.groupby('state')['policy_intensity_no_vam'].shift(1)
df['backlash_lag1'] = df.groupby('state')['backlash'].shift(1)
df['backlash_x_waiver_no_vam'] = df['backlash_lag1'] * df['has_waiver']

# Re-run H7b with Leave-One-Component-Out policy index
df_h7_no_vam = df[['state', 'year', 'delta_policy_no_vam', 'backlash_lag1', 'has_waiver', 'backlash_x_waiver_no_vam', 'gov_party_rep', 'trifecta']].dropna().set_index(['state', 'year'])
fit_h7b_no_vam = PanelOLS.from_formula(
    'delta_policy_no_vam ~ backlash_lag1 + has_waiver + backlash_x_waiver_no_vam + gov_party_rep + trifecta + EntityEffects + TimeEffects',
    data=df_h7_no_vam
).fit(cov_type='clustered', cluster_entity=True)

print('=== H7b Robustness: Leave-One-Component-Out (No VAM) ===')
print(fit_h7b_no_vam.summary.tables[1])

# 2. Individual outcome components separately
for comp in ['exit_exam', 'af_grading', 'third_grade_retention', 'vam_eval']:
    df[f'delta_{comp}'] = df.groupby('state')[comp].diff()
    df_h7_comp = df[['state', 'year', f'delta_{comp}', 'backlash_lag1', 'has_waiver', 'backlash_x_waiver', 'gov_party_rep', 'trifecta']].dropna().set_index(['state', 'year'])
    fit_comp = PanelOLS.from_formula(
        f'delta_{comp} ~ backlash_lag1 + has_waiver + backlash_x_waiver + gov_party_rep + trifecta + EntityEffects + TimeEffects',
        data=df_h7_comp
).fit(cov_type='clustered', cluster_entity=True)
    print(f'\n=== H7b Robustness: Outcome = Change in {comp} ===')
    print(fit_comp.summary.tables[1])

=== H7b Robustness: Leave-One-Component-Out (No VAM) ===
                                    Parameter Estimates                                     
                          Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
--------------------------------------------------------------------------------------------
backlash_lag1                0.0122     0.0249     0.4909     0.6237     -0.0366      0.0610
has_waiver                   0.0401     0.0398     1.0070     0.3143     -0.0381      0.1184
backlash_x_waiver_no_vam     0.0229     0.0360     0.6347     0.5258     -0.0479      0.0936
gov_party_rep               -0.0016     0.0335    -0.0483     0.9615     -0.0673      0.0641
trifecta                    -0.0001     0.0529    -0.0022     0.9983     -0.1041      0.1038

=== H7b Robustness: Outcome = Change in exit_exam ===
                                 Parameter Estimates                                 
                   Parameter  Std. Err.     T-stat    P-va

In [9]:
# 3. Disaggregated backlash specifications
for b_var in ['backlash_media', 'backlash_mass']:
    df[f'{b_var}_lag1'] = df.groupby('state')[b_var].shift(1)
    df[f'{b_var}_x_waiver'] = df[f'{b_var}_lag1'] * df['has_waiver']
    
    df_h7_b = df[['state', 'year', 'delta_policy', f'{b_var}_lag1', 'has_waiver', f'{b_var}_x_waiver', 'gov_party_rep', 'trifecta']].dropna().set_index(['state', 'year'])
    fit_b = PanelOLS.from_formula(
        f'delta_policy ~ {b_var}_lag1 + has_waiver + {b_var}_x_waiver + gov_party_rep + trifecta + EntityEffects + TimeEffects',
        data=df_h7_b
    ).fit(cov_type='clustered', cluster_entity=True)
    print(f'\n=== H7b Robustness: Predictor = Lagged {b_var} ===')
    print(fit_b.summary.tables[1])


=== H7b Robustness: Predictor = Lagged backlash_media ===


                                    Parameter Estimates                                    
                         Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
-------------------------------------------------------------------------------------------
backlash_media_lag1        -0.0044     0.0191    -0.2289     0.8190     -0.0419      0.0332
has_waiver                  0.5629     0.0806     6.9861     0.0000      0.4047      0.7211
backlash_media_x_waiver     0.0003     0.0263     0.0123     0.9902     -0.0514      0.0520
gov_party_rep               0.0331     0.0312     1.0605     0.2893     -0.0282      0.0943
trifecta                    0.0634     0.0447     1.4191     0.1563     -0.0243      0.1512

=== H7b Robustness: Predictor = Lagged backlash_mass ===
                                   Parameter Estimates                                    
                        Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
-----------------------

In [10]:
# 4. Conditional permutation placebo test (1,000 runs)
unique_states = df['state'].unique()
state_party = df.groupby('state')['gov_party_rep'].mean().to_dict()
state_strata = {}
for s, p in state_party.items():
    if p <= 0.2:
        state_strata[s] = 'Dem'
    elif p >= 0.8:
        state_strata[s] = 'Rep'
    else:
        state_strata[s] = 'Swing'

state_waivers = df.groupby('state')['has_waiver'].apply(list).to_dict()

placebo_coefs = []
np.random.seed(42)
for i in range(1000):
    shuffled_mapping = {}
    for strata_val in ['Dem', 'Rep', 'Swing']:
        strata_states = [s for s, v in state_strata.items() if v == strata_val]
        shuffled_strata = np.random.permutation(strata_states)
        for s_orig, s_shuf in zip(strata_states, shuffled_strata):
            shuffled_mapping[s_orig] = state_waivers[s_shuf]
            
    df_temp = df.copy()
    df_temp['has_waiver_placebo'] = df_temp.apply(lambda r: shuffled_mapping[r['state']][r['year'] - 2010], axis=1)
    df_temp['backlash_x_waiver_placebo'] = df_temp['backlash_lag1'] * df_temp['has_waiver_placebo']
    
    df_h7_temp = df_temp[['state', 'year', 'delta_policy', 'backlash_lag1', 'has_waiver_placebo', 'backlash_x_waiver_placebo', 'gov_party_rep', 'trifecta']].dropna().set_index(['state', 'year'])
    fit_placebo = PanelOLS.from_formula(
        'delta_policy ~ backlash_lag1 + has_waiver_placebo + backlash_x_waiver_placebo + gov_party_rep + trifecta + EntityEffects + TimeEffects',
        data=df_h7_temp
    ).fit(cov_type='clustered', cluster_entity=True)
    placebo_coefs.append(fit_placebo.params['backlash_x_waiver_placebo'])

placebo_coefs = np.array(placebo_coefs)
observed_coef = fit_h7b.params['backlash_x_waiver']
p_val = (1 + np.sum(np.abs(placebo_coefs) >= np.abs(observed_coef))) / 1001.0
print(f'\n=== H7b Robustness: Randomization Inference (1,000 Permutations) ===')
print(f'Observed Coefficient: {observed_coef:.4f}')
print(f'Mean Placebo Coefficient: {np.mean(placebo_coefs):.4f}')
print(f'Randomization p-value: {p_val:.4f}')


=== H7b Robustness: Randomization Inference (1,000 Permutations) ===
Observed Coefficient: -0.1685
Mean Placebo Coefficient: 0.0107
Randomization p-value: 0.0010


In [11]:
# 5. Clustered block bootstrap for H7b
bootstrap_h7b_coefs = []
np.random.seed(42)
for b in range(200):
    resampled_states = np.random.choice(unique_states, size=len(unique_states), replace=True)
    resampled_df_list = []
    for idx, s in enumerate(resampled_states):
        state_data = df[df['state'] == s].copy()
        state_data['bootstrap_id'] = f'{s}_{idx}'
        resampled_df_list.append(state_data)
    resampled_df = pd.concat(resampled_df_list, ignore_index=True)
    resampled_df['backlash_lag1'] = resampled_df.groupby('bootstrap_id')['backlash'].shift(1)
    resampled_df['delta_policy'] = resampled_df.groupby('bootstrap_id')['policy_intensity'].diff()
    resampled_df['backlash_x_waiver'] = resampled_df['backlash_lag1'] * resampled_df['has_waiver']
    
    df_boot_h7 = resampled_df[['bootstrap_id', 'year', 'delta_policy', 'backlash_lag1', 'has_waiver', 'backlash_x_waiver', 'gov_party_rep', 'trifecta']].dropna()
    fit_boot_h7 = smf.ols(
        'delta_policy ~ backlash_lag1 + has_waiver + backlash_x_waiver + gov_party_rep + trifecta + C(bootstrap_id) + C(year)',
        data=df_boot_h7
    ).fit()
    bootstrap_h7b_coefs.append(fit_boot_h7.params['backlash_x_waiver'])

bootstrap_h7b_coefs = np.array(bootstrap_h7b_coefs)
ci_lower_h7 = np.percentile(bootstrap_h7b_coefs, 2.5)
ci_upper_h7 = np.percentile(bootstrap_h7b_coefs, 97.5)
print(f'\n=== H7b Robustness: Clustered Block Bootstrap (200 Runs) ===')
print(f'backlash_x_waiver 95% Bootstrap CI: [{ci_lower_h7:.4f}, {ci_upper_h7:.4f}]')


=== H7b Robustness: Clustered Block Bootstrap (200 Runs) ===
backlash_x_waiver 95% Bootstrap CI: [-0.2511, -0.0869]


In [12]:
# 6. Policy-norm gap model (backlash ~ gap)
df['policy_lag1'] = df.groupby('state')['policy_intensity'].shift(1)
df['norm_lag1'] = df.groupby('state')['norm'].shift(1)
df['policy_gap_lag1'] = df['policy_lag1'] - df['norm_lag1']
df['abs_policy_gap_lag1'] = df['policy_gap_lag1'].abs()

# Decompose gap into positive and negative deviations
df['gap_pos_lag1'] = np.maximum(0, df['policy_gap_lag1'])
df['gap_neg_lag1'] = np.maximum(0, -df['policy_gap_lag1'])

df_gap = df[['state', 'year', 'backlash', 'backlash_lag1', 'abs_policy_gap_lag1', 'gap_pos_lag1', 'gap_neg_lag1', 'gov_party_rep', 'trifecta', 'election_year']].dropna().set_index(['state', 'year'])

# Standard absolute gap model (controlling for backlash_lag1 to prevent OVB)
fit_gap = PanelOLS.from_formula(
    'backlash ~ backlash_lag1 + abs_policy_gap_lag1 + gov_party_rep + trifecta + election_year + EntityEffects + TimeEffects',
    data=df_gap
).fit(cov_type='clustered', cluster_entity=True)
print(f'\n=== Policy-Norm Gap model: Absolute Gap ===')
print(fit_gap.summary.tables[1])

# Asymmetric gap model
fit_asym = PanelOLS.from_formula(
    'backlash ~ backlash_lag1 + gap_pos_lag1 + gap_neg_lag1 + gov_party_rep + trifecta + election_year + EntityEffects + TimeEffects',
    data=df_gap
).fit(cov_type='clustered', cluster_entity=True)
print(f'\n=== Policy-Norm Gap model: Asymmetric Gap ===')
print(fit_asym.summary.tables[1])

# Grid search for optimal theta (nested bootstrap in the visualization/analysis)
for th in [0.5, 1.0]:
    df[f'gap_th_{th}'] = np.maximum(0, df['abs_policy_gap_lag1'] - th)
    df_th = df[['state', 'year', 'backlash', 'backlash_lag1', f'gap_th_{th}', 'gov_party_rep', 'trifecta', 'election_year']].dropna().set_index(['state', 'year'])
    fit_th = PanelOLS.from_formula(
        f'backlash ~ backlash_lag1 + gap_th_{th} + gov_party_rep + trifecta + election_year + EntityEffects + TimeEffects',
        data=df_th
    ).fit(cov_type='clustered', cluster_entity=True)
    print(f'\n=== Policy-Norm Gap model: Threshold theta = {th} ===')
    print(fit_th.summary.tables[1])


=== Policy-Norm Gap model: Absolute Gap ===
                                  Parameter Estimates                                  
                     Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
---------------------------------------------------------------------------------------
backlash_lag1           0.4423     0.0818     5.4056     0.0000      0.2816      0.6029
abs_policy_gap_lag1    -0.0635     0.0897    -0.7080     0.4792     -0.2396      0.1126
gov_party_rep          -0.1882     0.0957    -1.9657     0.0498     -0.3761     -0.0002
trifecta               -0.7018     0.3436    -2.0426     0.0415     -1.3766     -0.0271
election_year           0.0691     0.0516     1.3412     0.1803     -0.0321      0.1704

=== Policy-Norm Gap model: Asymmetric Gap ===
                               Parameter Estimates                               
               Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
----------------------------------------

In [13]:
# 7. Correction model with mean reversion controls
df['correction'] = -1.0 * (df['policy_gap_lag1'] * df['delta_policy'])
df_corr = df[['state', 'year', 'correction', 'backlash_lag1', 'policy_gap_lag1', 'gov_party_rep', 'trifecta']].dropna().set_index(['state', 'year'])

# Run regression with lagged gap to control for natural mean reversion
fit_corr = PanelOLS.from_formula(
    'correction ~ backlash_lag1 + policy_gap_lag1 + gov_party_rep + trifecta + EntityEffects + TimeEffects',
    data=df_corr
).fit(cov_type='clustered', cluster_entity=True)
print(f'\n=== Policy Correction toward the Norm (with Mean Reversion controls) ===')
print(fit_corr.summary.tables[1])

# Distributed lag model of backlash on correction
for l in range(1, 4):
    df[f'backlash_lag{l}'] = df.groupby('state')['backlash'].shift(l)
    
df_dist = df[['state', 'year', 'correction', 'backlash_lag1', 'backlash_lag2', 'backlash_lag3', 'policy_gap_lag1', 'gov_party_rep', 'trifecta']].dropna().set_index(['state', 'year'])
fit_dist = PanelOLS.from_formula(
    'correction ~ backlash_lag1 + backlash_lag2 + backlash_lag3 + policy_gap_lag1 + gov_party_rep + trifecta + EntityEffects + TimeEffects',
    data=df_dist
).fit(cov_type='clustered', cluster_entity=True)
print(f'\n=== Distributed Lag Model of Backlash on Correction ===')
print(fit_dist.summary.tables[1])


=== Policy Correction toward the Norm (with Mean Reversion controls) ===
                                Parameter Estimates                                
                 Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
-----------------------------------------------------------------------------------
backlash_lag1      -0.0365     0.0215    -1.6983     0.0899     -0.0786      0.0057
policy_gap_lag1     0.0976     0.0214     4.5550     0.0000      0.0555      0.1397
gov_party_rep      -0.0384     0.0353    -1.0857     0.2780     -0.1077      0.0310
trifecta           -0.0567     0.0367    -1.5437     0.1232     -0.1288      0.0154

=== Distributed Lag Model of Backlash on Correction ===
                                Parameter Estimates                                
                 Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
-----------------------------------------------------------------------------------
backlash_lag1      -0.0244   

In [14]:
# 8. H2 oscillation delay vs amplitude (cross-sectional, descriptive)
# Exogenous delay proxy: Legislative session frequency (annual vs biennial)
df['biennial_legislature'] = df['state'].isin(['TX', 'MT', 'NV', 'ND']).astype(float)

# Detrend policy intensity for each state to isolate cycles
detrended_series = []
for state, gp in df.groupby('state'):
    gp = gp.sort_values('year').copy()
    y = gp['policy_intensity'].values
    x = gp['year'].values
    if len(gp) > 1:
        slope, intercept = np.polyfit(x, y, 1)
        gp['policy_detrended'] = y - (slope * x + intercept)
    else:
        gp['policy_detrended'] = 0.0
    detrended_series.append(gp)
df_detrend = pd.concat(detrended_series, ignore_index=True)

# Amplitude = Standard deviation of detrended policy intensity
state_amplitude = df_detrend.groupby('state')['policy_detrended'].std().reset_index().rename(columns={'policy_detrended': 'amplitude'})
state_biennial = df.groupby('state')['biennial_legislature'].first().reset_index()
df_h2 = pd.merge(state_amplitude, state_biennial, on='state')

# Regress amplitude on biennial dummy (descriptive check)
import statsmodels.api as sm
X = sm.add_constant(df_h2['biennial_legislature'])
fit_h2 = sm.OLS(df_h2['amplitude'], X).fit()
print(f'\n=== H2 descriptive: Amplitude (detrended SD) vs. Biennial Legislature (Delay Proxy) ===')
print(fit_h2.summary().tables[1])


=== H2 descriptive: Amplitude (detrended SD) vs. Biennial Legislature (Delay Proxy) ===
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                    0.4202      0.024     17.574      0.000       0.372       0.468
biennial_legislature    -0.1814      0.085     -2.125      0.039      -0.353      -0.010
